In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ziyadaltalhi/all-data-set/All Datasets For GP/Tik Tok Datasets - After Processing/المنطقة الوسطى/Tik Data/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-34-09-183_textready_analysis.xlsx
/kaggle/input/datasets/ziyadaltalhi/all-data-set/All Datasets For GP/Tik Tok Datasets - After Processing/المنطقة الوسطى/Tik Data/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-06-43-625_textready_analysis.xlsx
/kaggle/input/datasets/ziyadaltalhi/all-data-set/All Datasets For GP/Tik Tok Datasets - After Processing/المنطقة الوسطى/Tik Data/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-17-06-320_textready_cleaned.xlsx
/kaggle/input/datasets/ziyadaltalhi/all-data-set/All Datasets For GP/Tik Tok Datasets - After Processing/المنطقة الوسطى/Tik Data/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-37-52-643_textready_analysis.xlsx
/kaggle/input/datasets/ziyadaltalhi/all-data-set/All Datasets For GP/Tik Tok Datasets - After Processing/المنطقة الوسطى/Tik Data/القصيم/d

In [3]:
# =========================================================
# UNIFIED LSTM TRAINING PIPELINE
# Google Maps + TikTok + YouTube
# Final version based on actual dataset rules
# =========================================================

# =========================================
# 1) IMPORTS
# =========================================
import os
import json
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Dense,
    Dropout,
    Bidirectional,
    SpatialDropout1D,
    GlobalMaxPooling1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")


# =========================================
# 2) SETTINGS
# =========================================
ROOT_FOLDER = r"/kaggle/input/datasets/ziyadaltalhi/all-data-set/All Datasets For GP"
OUTPUT_DIR = r"/kaggle/working/final_unified_lstm_multisource"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42

TEXT_COL = "Text_TR"

MAX_WORDS = 50000
MAX_LEN = 100
EMBED_DIM = 128

BATCH_SIZE = 128
EPOCHS = 18
LEARNING_RATE = 3e-4

USE_CLASS_WEIGHTS = True
BOOST_NEGATIVE_WEIGHT = True
NEGATIVE_BOOST_FACTOR = 1.10

DOWNSAMPLE_NEUTRAL = True
NEUTRAL_MAX = 120000   # تقدر تعدله لاحقًا إذا احتجت

MIN_WORDS = 2

EXCLUDED_COMMENT_TYPES = {
    "noise",
    "noisy",
    "ad",
    "ads",
    "question",
    "questions",
    "channel_comment"
}


# =========================================
# 3) SEED
# =========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(RANDOM_STATE)


# =========================================
# 4) HELPERS
# =========================================
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    x = " ".join(x.split())
    return x

def read_excel_safe(path):
    return pd.read_excel(path)

def build_label_from_stars(row):
    # Google غالبًا stars
    if "stars" in row and pd.notna(row["stars"]):
        try:
            val = float(row["stars"])
            if val >= 4:
                return "positive"
            elif val <= 2:
                return "negative"
            else:
                return "neutral"
        except:
            pass

    # TikTok / YouTube غالبًا Stars
    if "Stars" in row and pd.notna(row["Stars"]):
        try:
            val = float(row["Stars"])
            if val >= 4:
                return "positive"
            elif val <= 2:
                return "negative"
            else:
                return "neutral"
        except:
            pass

    return None

def encode_texts(texts, tokenizer, max_len):
    seq = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")


# =========================================
# 5) FIND FILES
# =========================================
root = Path(ROOT_FOLDER)

# Google
google_files = sorted([
    p for p in root.rglob("*_textready.xlsx")
    if "Cleaned Data From Google Maps" in str(p)
])

# TikTok cleaned only
tiktok_files = sorted([
    p for p in root.rglob("*_textready_cleaned.xlsx")
    if "Tik Tok Datasets - After Processing" in str(p)
])

# YouTube cleaned only
youtube_files = sorted([
    p for p in root.rglob("*_textready_cleaned.xlsx")
    if "YouTube Datasets - After Processing" in str(p)
])

print("=" * 70)
print("Google files found :", len(google_files))
print("TikTok files found :", len(tiktok_files))
print("YouTube files found:", len(youtube_files))
print("=" * 70)

if len(google_files) == 0 and len(tiktok_files) == 0 and len(youtube_files) == 0:
    raise ValueError("No valid files found. Check ROOT_FOLDER.")


# =========================================
# 6) LOAD GOOGLE
# =========================================
google_dfs = []
google_bad = []

for f in google_files:
    try:
        temp = read_excel_safe(f)
        temp["source"] = "google_maps"
        temp["Source_File"] = f.name
        temp["Source_Path"] = str(f)
        temp["Parent_Folder"] = f.parent.name
        google_dfs.append(temp)
    except Exception as e:
        google_bad.append((str(f), str(e)))

google_df = pd.concat(google_dfs, ignore_index=True) if google_dfs else pd.DataFrame()
print("Google shape before filtering:", google_df.shape)


# =========================================
# 7) LOAD TIKTOK
# =========================================
tiktok_dfs = []
tiktok_bad = []

for f in tiktok_files:
    try:
        temp = read_excel_safe(f)
        temp["source"] = "tiktok"
        temp["Source_File"] = f.name
        temp["Source_Path"] = str(f)
        temp["Parent_Folder"] = f.parent.name
        tiktok_dfs.append(temp)
    except Exception as e:
        tiktok_bad.append((str(f), str(e)))

tiktok_df = pd.concat(tiktok_dfs, ignore_index=True) if tiktok_dfs else pd.DataFrame()
print("TikTok shape before filtering:", tiktok_df.shape)


# =========================================
# 8) LOAD YOUTUBE
# =========================================
youtube_dfs = []
youtube_bad = []

for f in youtube_files:
    try:
        temp = read_excel_safe(f)
        temp["source"] = "youtube"
        temp["Source_File"] = f.name
        temp["Source_Path"] = str(f)
        temp["Parent_Folder"] = f.parent.name
        youtube_dfs.append(temp)
    except Exception as e:
        youtube_bad.append((str(f), str(e)))

youtube_df = pd.concat(youtube_dfs, ignore_index=True) if youtube_dfs else pd.DataFrame()
print("YouTube shape before filtering:", youtube_df.shape)


# =========================================
# 9) SAVE FAILED FILES
# =========================================
failed_rows = []
for f, e in google_bad:
    failed_rows.append(["google_maps", f, e])
for f, e in tiktok_bad:
    failed_rows.append(["tiktok", f, e])
for f, e in youtube_bad:
    failed_rows.append(["youtube", f, e])

if failed_rows:
    pd.DataFrame(failed_rows, columns=["source", "file", "error"]).to_excel(
        os.path.join(OUTPUT_DIR, "failed_files.xlsx"),
        index=False
    )


# =========================================
# 10) FILTER COMMENT TYPES
# TikTok / YouTube only
# =========================================
def filter_comment_types(df, source_name):
    if df.empty:
        return df

    before = len(df)

    if source_name in ["tiktok", "youtube"] and "comment_type" in df.columns:
        ct = (
            df["comment_type"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
        )
        df = df[~ct.isin(EXCLUDED_COMMENT_TYPES)].copy()

    after = len(df)
    print(f"{source_name}: {before} -> {after} after comment_type filtering")
    return df

tiktok_df = filter_comment_types(tiktok_df, "tiktok")
youtube_df = filter_comment_types(youtube_df, "youtube")


# =========================================
# 11) MERGE ALL
# =========================================
frames = [x for x in [google_df, tiktok_df, youtube_df] if not x.empty]

if not frames:
    raise ValueError("No data left after loading/filtering.")

df = pd.concat(frames, ignore_index=True)
print("Merged shape:", df.shape)


# =========================================
# 12) KEEP ONLY Text_TR
# =========================================
if TEXT_COL not in df.columns:
    raise ValueError(f"{TEXT_COL} column was not found in merged data.")

df[TEXT_COL] = df[TEXT_COL].apply(normalize_text)

before_text = len(df)
df = df[df[TEXT_COL] != ""].copy()
df["word_count"] = df[TEXT_COL].astype(str).str.split().str.len()
df = df[df["word_count"] >= MIN_WORDS].copy()
after_text = len(df)

print(f"After Text_TR filtering: {before_text} -> {after_text}")


# =========================================
# 13) BUILD LABEL FROM stars / Stars ONLY
# =========================================
before_label = len(df)
df["label"] = df.apply(build_label_from_stars, axis=1)
df = df.dropna(subset=["label"]).copy()
after_label = len(df)

print(f"After label building: {before_label} -> {after_label}")

print("\nLabel distribution before balancing:")
print(df["label"].value_counts())

print("\nSource distribution:")
print(df["source"].value_counts())

print("\nSource x Label distribution:")
print(pd.crosstab(df["source"], df["label"]))


# =========================================
# 14) OPTIONAL DOWNSAMPLE FOR NEUTRAL
# =========================================
if DOWNSAMPLE_NEUTRAL:
    neutral_df = df[df["label"] == "neutral"].copy()
    positive_df = df[df["label"] == "positive"].copy()
    negative_df = df[df["label"] == "negative"].copy()

    if len(neutral_df) > NEUTRAL_MAX:
        neutral_df = neutral_df.sample(n=NEUTRAL_MAX, random_state=RANDOM_STATE)

    df = pd.concat([positive_df, negative_df, neutral_df], ignore_index=True)
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("\nLabel distribution after balancing:")
print(df["label"].value_counts())

print("\nFinal source x label distribution:")
print(pd.crosstab(df["source"], df["label"]))


# =========================================
# 15) ENCODE LABELS
# =========================================
le = LabelEncoder()
df["y"] = le.fit_transform(df["label"])

print("\nEncoded classes:")
for cls_name, cls_id in zip(le.classes_, le.transform(le.classes_)):
    print(f"{cls_id} -> {cls_name}")


# =========================================
# 16) STRATIFIED SPLIT
# =========================================
df["stratify_key"] = df["source"].astype(str) + "__" + df["label"].astype(str)

group_counts = df["stratify_key"].value_counts()
rare_keys = group_counts[group_counts < 3].index.tolist()

if rare_keys:
    rare_df = df[df["stratify_key"].isin(rare_keys)].copy()
    main_df = df[~df["stratify_key"].isin(rare_keys)].copy()
    print(f"\nRare groups moved to train only: {len(rare_df)}")
else:
    rare_df = pd.DataFrame()
    main_df = df.copy()

X = main_df[TEXT_COL].astype(str).tolist()
y = main_df["y"].values
idx = main_df.index.values
stratify_values = main_df["stratify_key"].values

X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(
    X,
    y,
    idx,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=stratify_values
)

temp_df = main_df.loc[idx_temp].copy()

X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(
    temp_df[TEXT_COL].astype(str).tolist(),
    temp_df["y"].values,
    temp_df.index.values,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["stratify_key"].values
)

if not rare_df.empty:
    X_train = X_train + rare_df[TEXT_COL].astype(str).tolist()
    y_train = np.concatenate([y_train, rare_df["y"].values])
    idx_train = np.concatenate([idx_train, rare_df.index.values])

print("\nTrain size:", len(X_train))
print("Val size  :", len(X_val))
print("Test size :", len(X_test))


# =========================================
# 17) TOKENIZATION
# =========================================
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_pad = encode_texts(X_train, tokenizer, MAX_LEN)
X_val_pad   = encode_texts(X_val, tokenizer, MAX_LEN)
X_test_pad  = encode_texts(X_test, tokenizer, MAX_LEN)

print("\nEncoded shapes:")
print("X_train_pad:", X_train_pad.shape)
print("X_val_pad  :", X_val_pad.shape)
print("X_test_pad :", X_test_pad.shape)


# =========================================
# 18) CLASS WEIGHTS
# =========================================
class_weight_dict = None

if USE_CLASS_WEIGHTS:
    classes = np.unique(y_train)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}

    if BOOST_NEGATIVE_WEIGHT and "negative" in le.classes_:
        neg_id = int(le.transform(["negative"])[0])
        class_weight_dict[neg_id] *= NEGATIVE_BOOST_FACTOR

    print("\nClass weights:")
    print(class_weight_dict)


# =========================================
# 19) MODEL
# =========================================
num_classes = len(le.classes_)

model = Sequential([
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=EMBED_DIM,
        input_length=MAX_LEN
    ),

    SpatialDropout1D(0.30),

    Bidirectional(
        LSTM(
            64,
            return_sequences=True,
            dropout=0.22,
            recurrent_dropout=0.20,
            kernel_regularizer=l2(1e-4),
            recurrent_regularizer=l2(1e-4)
        )
    ),

    Bidirectional(
        LSTM(
            32,
            return_sequences=True,
            dropout=0.20,
            recurrent_dropout=0.20,
            kernel_regularizer=l2(8e-5),
            recurrent_regularizer=l2(8e-5)
        )
    ),

    GlobalMaxPooling1D(),

    Dense(64, activation="relu", kernel_regularizer=l2(1e-4)),
    Dropout(0.45),

    Dense(32, activation="relu", kernel_regularizer=l2(8e-5)),
    Dropout(0.25),

    Dense(num_classes, activation="softmax")
])

optimizer = Adam(learning_rate=LEARNING_RATE, clipnorm=1.0)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

model.summary()


# =========================================
# 20) CALLBACKS
# =========================================
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, "best_model.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]


# =========================================
# 21) TRAIN
# =========================================
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)


# =========================================
# 22) EVALUATE
# =========================================
y_prob = model.predict(X_test_pad, batch_size=BATCH_SIZE, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print("\n" + "=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)
print("Accuracy    :", acc)
print("F1 Macro    :", f1_macro)
print("F1 Weighted :", f1_weighted)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n")
print(cm)


# =========================================
# 23) SAVE REPORTS
# =========================================
report_dict = classification_report(
    y_test,
    y_pred,
    target_names=le.classes_,
    output_dict=True
)

pd.DataFrame(report_dict).transpose().to_excel(
    os.path.join(OUTPUT_DIR, "classification_report.xlsx")
)

pd.DataFrame(cm, index=le.classes_, columns=le.classes_).to_excel(
    os.path.join(OUTPUT_DIR, "confusion_matrix.xlsx")
)

pd.DataFrame(history.history).to_excel(
    os.path.join(OUTPUT_DIR, "training_history.xlsx"),
    index=False
)


# =========================================
# 24) SAVE TEST PREDICTIONS
# =========================================
test_df = df.loc[idx_test].copy().reset_index(drop=True)
test_df["y_true"] = y_test
test_df["y_pred"] = y_pred
test_df["true_label"] = le.inverse_transform(y_test)
test_df["pred_label"] = le.inverse_transform(y_pred)
test_df["pred_confidence"] = y_prob.max(axis=1)

for i, cls_name in enumerate(le.classes_):
    test_df[f"prob_{cls_name}"] = y_prob[:, i]

test_df.to_excel(
    os.path.join(OUTPUT_DIR, "test_predictions.xlsx"),
    index=False
)

errors_df = test_df[test_df["y_true"] != test_df["y_pred"]].copy()
errors_df = errors_df.sort_values("pred_confidence", ascending=False)
errors_df.to_excel(
    os.path.join(OUTPUT_DIR, "error_analysis.xlsx"),
    index=False
)


# =========================================
# 25) SAVE EXTRA STATS
# =========================================
pd.crosstab(df["source"], df["label"]).to_excel(
    os.path.join(OUTPUT_DIR, "source_label_distribution.xlsx")
)

pd.DataFrame({
    "source": df["source"].value_counts().index,
    "count": df["source"].value_counts().values
}).to_excel(
    os.path.join(OUTPUT_DIR, "source_distribution.xlsx"),
    index=False
)


# =========================================
# 26) SAVE MODEL + TOKENIZER + ENCODER
# =========================================
model.save(os.path.join(OUTPUT_DIR, "final_model.keras"))

with open(os.path.join(OUTPUT_DIR, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

with open(os.path.join(OUTPUT_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(le, f)


# =========================================
# 27) SAVE CONFIG
# =========================================
config = {
    "ROOT_FOLDER": ROOT_FOLDER,
    "OUTPUT_DIR": OUTPUT_DIR,
    "TEXT_COL": TEXT_COL,
    "MAX_WORDS": MAX_WORDS,
    "MAX_LEN": MAX_LEN,
    "EMBED_DIM": EMBED_DIM,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "USE_CLASS_WEIGHTS": USE_CLASS_WEIGHTS,
    "BOOST_NEGATIVE_WEIGHT": BOOST_NEGATIVE_WEIGHT,
    "NEGATIVE_BOOST_FACTOR": NEGATIVE_BOOST_FACTOR,
    "DOWNSAMPLE_NEUTRAL": DOWNSAMPLE_NEUTRAL,
    "NEUTRAL_MAX": NEUTRAL_MAX,
    "MIN_WORDS": MIN_WORDS,
    "EXCLUDED_COMMENT_TYPES": sorted(list(EXCLUDED_COMMENT_TYPES)),
    "classes": list(le.classes_),
    "accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "f1_weighted": float(f1_weighted),
    "final_shape": list(df.shape),
    "google_files": len(google_files),
    "tiktok_files": len(tiktok_files),
    "youtube_files": len(youtube_files),
    "label_distribution": {
        str(k): int(v) for k, v in df["label"].value_counts().to_dict().items()
    },
    "source_distribution": {
        str(k): int(v) for k, v in df["source"].value_counts().to_dict().items()
    }
}

with open(os.path.join(OUTPUT_DIR, "config_summary.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)


# =========================================
# 28) FINAL PRINT
# =========================================
print("\nDone successfully.")
print(json.dumps({
    "accuracy": acc,
    "f1_macro": f1_macro,
    "f1_weighted": f1_weighted
}, ensure_ascii=False, indent=2))

Google files found : 442
TikTok files found : 295
YouTube files found: 198
Google shape before filtering: (1016595, 84)
TikTok shape before filtering: (113571, 80)
YouTube shape before filtering: (58175, 38)
tiktok: 113571 -> 44128 after comment_type filtering
youtube: 58175 -> 37568 after comment_type filtering
Merged shape: (1098291, 153)
After Text_TR filtering: 1098291 -> 539648
After label building: 539648 -> 539366

Label distribution before balancing:
label
positive    387933
negative     82752
neutral      68681
Name: count, dtype: int64

Source distribution:
source
google_maps    457757
tiktok          44080
youtube         37529
Name: count, dtype: int64

Source x Label distribution:
label        negative  neutral  positive
source                                  
google_maps     57053    49683    351021
tiktok          13728    13010     17342
youtube         11971     5988     19570

Label distribution after balancing:
label
positive    387933
negative     82752
neutral    

I0000 00:00:1776286962.773391      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776286962.779681      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/18
 799/3372 ━━━━━━━━━━━━━━━━━━━━ 46:32 1s/step - accuracy: 0.3606 - loss: 1.1286

KeyboardInterrupt: 

In [1]:
# =========================================================
# UNIFIED BINARY LSTM TRAINING PIPELINE
# Google Maps + TikTok + YouTube
# positive / negative only
# =========================================================

# =========================================
# 1) IMPORTS
# =========================================
import os
import json
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Dense,
    Dropout,
    Bidirectional,
    SpatialDropout1D,
    GlobalMaxPooling1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")


# =========================================
# 2) SETTINGS
# =========================================
ROOT_FOLDER = r"/kaggle/input/datasets/ziyadaltalhi/all-data-set/All Datasets For GP"
OUTPUT_DIR = r"/kaggle/working/final_unified_lstm_binary"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
TEXT_COL = "Text_TR"

MAX_WORDS = 50000
MAX_LEN = 100
EMBED_DIM = 128

BATCH_SIZE = 128
EPOCHS = 18
LEARNING_RATE = 3e-4

USE_CLASS_WEIGHTS = True
BOOST_NEGATIVE_WEIGHT = True
NEGATIVE_BOOST_FACTOR = 1.12

# مهم:
# نسمح بالنصوص القصيرة جدًا طالما ليست فارغة
# لذلك لن نستخدم شرط MIN_WORDS >= 2
REMOVE_EMPTY_TEXT_ONLY = True

EXCLUDED_COMMENT_TYPES = {
    "noise",
    "noisy",
    "ad",
    "ads",
    "question",
    "questions",
    "channel_comment"
}


# =========================================
# 3) SEED
# =========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(RANDOM_STATE)


# =========================================
# 4) HELPERS
# =========================================
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    x = " ".join(x.split())
    return x

def read_excel_safe(path):
    return pd.read_excel(path)

def build_binary_label_from_stars(row):
    # Google: غالبًا stars
    if "stars" in row and pd.notna(row["stars"]):
        try:
            val = float(row["stars"])
            if val >= 4:
                return "positive"
            elif val <= 2:
                return "negative"
            else:
                return None   # نحذف المحايد
        except:
            pass

    # TikTok / YouTube: غالبًا Stars
    if "Stars" in row and pd.notna(row["Stars"]):
        try:
            val = float(row["Stars"])
            if val >= 4:
                return "positive"
            elif val <= 2:
                return "negative"
            else:
                return None   # نحذف المحايد
        except:
            pass

    return None

def encode_texts(texts, tokenizer, max_len):
    seq = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")


# =========================================
# 5) FIND FILES
# =========================================
root = Path(ROOT_FOLDER)

google_files = sorted([
    p for p in root.rglob("*_textready.xlsx")
    if "Cleaned Data From Google Maps" in str(p)
])

tiktok_files = sorted([
    p for p in root.rglob("*_textready_cleaned.xlsx")
    if "Tik Tok Datasets - After Processing" in str(p)
])

youtube_files = sorted([
    p for p in root.rglob("*_textready_cleaned.xlsx")
    if "YouTube Datasets - After Processing" in str(p)
])

print("=" * 70)
print("Google files found :", len(google_files))
print("TikTok files found :", len(tiktok_files))
print("YouTube files found:", len(youtube_files))
print("=" * 70)

if len(google_files) == 0 and len(tiktok_files) == 0 and len(youtube_files) == 0:
    raise ValueError("No valid files found. Check ROOT_FOLDER.")


# =========================================
# 6) LOAD GOOGLE
# =========================================
google_dfs = []
google_bad = []

for f in google_files:
    try:
        temp = read_excel_safe(f)
        temp["source"] = "google_maps"
        temp["Source_File"] = f.name
        temp["Source_Path"] = str(f)
        temp["Parent_Folder"] = f.parent.name
        google_dfs.append(temp)
    except Exception as e:
        google_bad.append((str(f), str(e)))

google_df = pd.concat(google_dfs, ignore_index=True) if google_dfs else pd.DataFrame()
print("Google shape before filtering:", google_df.shape)


# =========================================
# 7) LOAD TIKTOK
# =========================================
tiktok_dfs = []
tiktok_bad = []

for f in tiktok_files:
    try:
        temp = read_excel_safe(f)
        temp["source"] = "tiktok"
        temp["Source_File"] = f.name
        temp["Source_Path"] = str(f)
        temp["Parent_Folder"] = f.parent.name
        tiktok_dfs.append(temp)
    except Exception as e:
        tiktok_bad.append((str(f), str(e)))

tiktok_df = pd.concat(tiktok_dfs, ignore_index=True) if tiktok_dfs else pd.DataFrame()
print("TikTok shape before filtering:", tiktok_df.shape)


# =========================================
# 8) LOAD YOUTUBE
# =========================================
youtube_dfs = []
youtube_bad = []

for f in youtube_files:
    try:
        temp = read_excel_safe(f)
        temp["source"] = "youtube"
        temp["Source_File"] = f.name
        temp["Source_Path"] = str(f)
        temp["Parent_Folder"] = f.parent.name
        youtube_dfs.append(temp)
    except Exception as e:
        youtube_bad.append((str(f), str(e)))

youtube_df = pd.concat(youtube_dfs, ignore_index=True) if youtube_dfs else pd.DataFrame()
print("YouTube shape before filtering:", youtube_df.shape)


# =========================================
# 9) SAVE FAILED FILES
# =========================================
failed_rows = []
for f, e in google_bad:
    failed_rows.append(["google_maps", f, e])
for f, e in tiktok_bad:
    failed_rows.append(["tiktok", f, e])
for f, e in youtube_bad:
    failed_rows.append(["youtube", f, e])

if failed_rows:
    pd.DataFrame(failed_rows, columns=["source", "file", "error"]).to_excel(
        os.path.join(OUTPUT_DIR, "failed_files.xlsx"),
        index=False
    )


# =========================================
# 10) FILTER COMMENT TYPES
# TikTok / YouTube only
# =========================================
def filter_comment_types(df, source_name):
    if df.empty:
        return df

    before = len(df)

    if source_name in ["tiktok", "youtube"] and "comment_type" in df.columns:
        ct = (
            df["comment_type"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
        )
        df = df[~ct.isin(EXCLUDED_COMMENT_TYPES)].copy()

    after = len(df)
    print(f"{source_name}: {before} -> {after} after comment_type filtering")
    return df

tiktok_df = filter_comment_types(tiktok_df, "tiktok")
youtube_df = filter_comment_types(youtube_df, "youtube")


# =========================================
# 11) MERGE ALL
# =========================================
frames = [x for x in [google_df, tiktok_df, youtube_df] if not x.empty]

if not frames:
    raise ValueError("No data left after loading/filtering.")

df = pd.concat(frames, ignore_index=True)
print("Merged shape:", df.shape)


# =========================================
# 12) KEEP ONLY Text_TR
# نحذف الفارغ فقط - ولا نحذف القصير
# =========================================
if TEXT_COL not in df.columns:
    raise ValueError(f"{TEXT_COL} column was not found in merged data.")

df[TEXT_COL] = df[TEXT_COL].apply(normalize_text)

before_text = len(df)
if REMOVE_EMPTY_TEXT_ONLY:
    df = df[df[TEXT_COL] != ""].copy()
after_text = len(df)

print(f"After Text_TR non-empty filtering: {before_text} -> {after_text}")


# =========================================
# 13) BUILD BINARY LABEL FROM stars / Stars ONLY
# positive / negative فقط
# =========================================
before_label = len(df)
df["label"] = df.apply(build_binary_label_from_stars, axis=1)
df = df.dropna(subset=["label"]).copy()
after_label = len(df)

print(f"After binary label building: {before_label} -> {after_label}")

print("\nBinary label distribution:")
print(df["label"].value_counts())

print("\nSource distribution:")
print(df["source"].value_counts())

print("\nSource x Binary Label distribution:")
print(pd.crosstab(df["source"], df["label"]))


# =========================================
# 14) ENCODE LABELS
# =========================================
le = LabelEncoder()
df["y"] = le.fit_transform(df["label"])

print("\nEncoded classes:")
for cls_name, cls_id in zip(le.classes_, le.transform(le.classes_)):
    print(f"{cls_id} -> {cls_name}")


# =========================================
# 15) STRATIFIED SPLIT
# =========================================
df["stratify_key"] = df["source"].astype(str) + "__" + df["label"].astype(str)

group_counts = df["stratify_key"].value_counts()
rare_keys = group_counts[group_counts < 3].index.tolist()

if rare_keys:
    rare_df = df[df["stratify_key"].isin(rare_keys)].copy()
    main_df = df[~df["stratify_key"].isin(rare_keys)].copy()
    print(f"\nRare groups moved to train only: {len(rare_df)}")
else:
    rare_df = pd.DataFrame()
    main_df = df.copy()

X = main_df[TEXT_COL].astype(str).tolist()
y = main_df["y"].values
idx = main_df.index.values
stratify_values = main_df["stratify_key"].values

X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(
    X,
    y,
    idx,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=stratify_values
)

temp_df = main_df.loc[idx_temp].copy()

X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(
    temp_df[TEXT_COL].astype(str).tolist(),
    temp_df["y"].values,
    temp_df.index.values,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["stratify_key"].values
)

if not rare_df.empty:
    X_train = X_train + rare_df[TEXT_COL].astype(str).tolist()
    y_train = np.concatenate([y_train, rare_df["y"].values])
    idx_train = np.concatenate([idx_train, rare_df.index.values])

print("\nTrain size:", len(X_train))
print("Val size  :", len(X_val))
print("Test size :", len(X_test))


# =========================================
# 16) TOKENIZATION
# =========================================
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_pad = encode_texts(X_train, tokenizer, MAX_LEN)
X_val_pad   = encode_texts(X_val, tokenizer, MAX_LEN)
X_test_pad  = encode_texts(X_test, tokenizer, MAX_LEN)

print("\nEncoded shapes:")
print("X_train_pad:", X_train_pad.shape)
print("X_val_pad  :", X_val_pad.shape)
print("X_test_pad :", X_test_pad.shape)


# =========================================
# 17) CLASS WEIGHTS
# =========================================
class_weight_dict = None

if USE_CLASS_WEIGHTS:
    classes = np.unique(y_train)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}

    if BOOST_NEGATIVE_WEIGHT and "negative" in le.classes_:
        neg_id = int(le.transform(["negative"])[0])
        class_weight_dict[neg_id] *= NEGATIVE_BOOST_FACTOR

    print("\nClass weights:")
    print(class_weight_dict)


# =========================================
# 18) MODEL
# =========================================
num_classes = len(le.classes_)

model = Sequential([
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=EMBED_DIM,
        input_length=MAX_LEN
    ),

    SpatialDropout1D(0.30),

    Bidirectional(
        LSTM(
            64,
            return_sequences=True,
            dropout=0.22,
            recurrent_dropout=0.20,
            kernel_regularizer=l2(1e-4),
            recurrent_regularizer=l2(1e-4)
        )
    ),

    Bidirectional(
        LSTM(
            32,
            return_sequences=True,
            dropout=0.20,
            recurrent_dropout=0.20,
            kernel_regularizer=l2(8e-5),
            recurrent_regularizer=l2(8e-5)
        )
    ),

    GlobalMaxPooling1D(),

    Dense(64, activation="relu", kernel_regularizer=l2(1e-4)),
    Dropout(0.45),

    Dense(32, activation="relu", kernel_regularizer=l2(8e-5)),
    Dropout(0.25),

    Dense(num_classes, activation="softmax")
])

optimizer = Adam(learning_rate=LEARNING_RATE, clipnorm=1.0)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

model.summary()


# =========================================
# 19) CALLBACKS
# =========================================
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, "best_model.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]


# =========================================
# 20) TRAIN
# =========================================
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)


# =========================================
# 21) EVALUATE
# =========================================
y_prob = model.predict(X_test_pad, batch_size=BATCH_SIZE, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print("\n" + "=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)
print("Accuracy    :", acc)
print("F1 Macro    :", f1_macro)
print("F1 Weighted :", f1_weighted)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n")
print(cm)


# =========================================
# 22) SAVE REPORTS
# =========================================
report_dict = classification_report(
    y_test,
    y_pred,
    target_names=le.classes_,
    output_dict=True
)

pd.DataFrame(report_dict).transpose().to_excel(
    os.path.join(OUTPUT_DIR, "classification_report.xlsx")
)

pd.DataFrame(cm, index=le.classes_, columns=le.classes_).to_excel(
    os.path.join(OUTPUT_DIR, "confusion_matrix.xlsx")
)

pd.DataFrame(history.history).to_excel(
    os.path.join(OUTPUT_DIR, "training_history.xlsx"),
    index=False
)


# =========================================
# 23) SAVE TEST PREDICTIONS
# =========================================
test_df = df.loc[idx_test].copy().reset_index(drop=True)
test_df["y_true"] = y_test
test_df["y_pred"] = y_pred
test_df["true_label"] = le.inverse_transform(y_test)
test_df["pred_label"] = le.inverse_transform(y_pred)
test_df["pred_confidence"] = y_prob.max(axis=1)

for i, cls_name in enumerate(le.classes_):
    test_df[f"prob_{cls_name}"] = y_prob[:, i]

test_df.to_excel(
    os.path.join(OUTPUT_DIR, "test_predictions.xlsx"),
    index=False
)

errors_df = test_df[test_df["y_true"] != test_df["y_pred"]].copy()
errors_df = errors_df.sort_values("pred_confidence", ascending=False)
errors_df.to_excel(
    os.path.join(OUTPUT_DIR, "error_analysis.xlsx"),
    index=False
)


# =========================================
# 24) SAVE EXTRA STATS
# =========================================
pd.crosstab(df["source"], df["label"]).to_excel(
    os.path.join(OUTPUT_DIR, "source_label_distribution.xlsx")
)

pd.DataFrame({
    "source": df["source"].value_counts().index,
    "count": df["source"].value_counts().values
}).to_excel(
    os.path.join(OUTPUT_DIR, "source_distribution.xlsx"),
    index=False
)


# =========================================
# 25) SAVE MODEL + TOKENIZER + ENCODER
# =========================================
model.save(os.path.join(OUTPUT_DIR, "final_model.keras"))

with open(os.path.join(OUTPUT_DIR, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

with open(os.path.join(OUTPUT_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(le, f)


# =========================================
# 26) SAVE CONFIG
# =========================================
config = {
    "ROOT_FOLDER": ROOT_FOLDER,
    "OUTPUT_DIR": OUTPUT_DIR,
    "TEXT_COL": TEXT_COL,
    "MAX_WORDS": MAX_WORDS,
    "MAX_LEN": MAX_LEN,
    "EMBED_DIM": EMBED_DIM,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "USE_CLASS_WEIGHTS": USE_CLASS_WEIGHTS,
    "BOOST_NEGATIVE_WEIGHT": BOOST_NEGATIVE_WEIGHT,
    "NEGATIVE_BOOST_FACTOR": NEGATIVE_BOOST_FACTOR,
    "REMOVE_EMPTY_TEXT_ONLY": REMOVE_EMPTY_TEXT_ONLY,
    "EXCLUDED_COMMENT_TYPES": sorted(list(EXCLUDED_COMMENT_TYPES)),
    "classes": list(le.classes_),
    "accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "f1_weighted": float(f1_weighted),
    "final_shape": list(df.shape),
    "google_files": len(google_files),
    "tiktok_files": len(tiktok_files),
    "youtube_files": len(youtube_files),
    "label_distribution": {
        str(k): int(v) for k, v in df["label"].value_counts().to_dict().items()
    },
    "source_distribution": {
        str(k): int(v) for k, v in df["source"].value_counts().to_dict().items()
    }
}

with open(os.path.join(OUTPUT_DIR, "config_summary.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)


# =========================================
# 27) FINAL PRINT
# =========================================
print("\nDone successfully.")
print(json.dumps({
    "accuracy": acc,
    "f1_macro": f1_macro,
    "f1_weighted": f1_weighted
}, ensure_ascii=False, indent=2))

2026-04-16 04:27:19.200761: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776313639.367363      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776313639.418774      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776313639.860817      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776313639.860853      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776313639.860859      55 computation_placer.cc:177] computation placer alr

Google files found : 442
TikTok files found : 295
YouTube files found: 198
Google shape before filtering: (1016595, 84)
TikTok shape before filtering: (113571, 80)
YouTube shape before filtering: (58175, 38)
tiktok: 113571 -> 44128 after comment_type filtering
youtube: 58175 -> 37568 after comment_type filtering
Merged shape: (1098291, 153)
After Text_TR non-empty filtering: 1098291 -> 593237
After binary label building: 593237 -> 518831

Binary label distribution:
label
positive    432185
negative     86646
Name: count, dtype: int64

Source distribution:
source
google_maps    456176
youtube         31562
tiktok          31093
Name: count, dtype: int64

Source x Binary Label distribution:
label        negative  positive
source                         
google_maps     60935    395241
tiktok          13738     17355
youtube         11973     19589

Encoded classes:
0 -> negative
1 -> positive

Train size: 415064
Val size  : 51883
Test size : 51884

Encoded shapes:
X_train_pad: (415064, 1

I0000 00:00:1776314070.075571      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776314070.081699      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/18
3243/3243 ━━━━━━━━━━━━━━━━━━━━ 0s 923ms/step - accuracy: 0.7715 - loss: 0.4957
Epoch 1: val_loss improved from inf to 0.32949, saving model to /kaggle/working/final_unified_lstm_binary/best_model.keras
3243/3243 ━━━━━━━━━━━━━━━━━━━━ 3077s 944ms/step - accuracy: 0.7715 - loss: 0.4956 - val_accuracy: 0.8840 - val_loss: 0.3295 - learning_rate: 3.0000e-04
Epoch 2/18
3243/3243 ━━━━━━━━━━━━━━━━━━━━ 0s 915ms/step - accuracy: 0.8936 - loss: 0.3199
Epoch 2: val_loss improved from 0.32949 to 0.32360, saving model to /kaggle/working/final_unified_lstm_binary/best_model.keras
3243/3243 ━━━━━━━━━━━━━━━━━━━━ 3035s 936ms/step - accuracy: 0.8936 - loss: 0.3199 - val_accuracy: 0.8897 - val_loss: 0.3236 - learning_rate: 3.0000e-04
Epoch 3/18
3243/3243 ━━━━━━━━━━━━━━━━━━━━ 0s 918ms/step - accuracy: 0.9042 - loss: 0.2884
Epoch 3: val_loss improved from 0.32360 to 0.31768, saving model to /kaggle/working/final_unified_lstm_binary/best_model.keras
3243/3243 ━━━━━━━━━━━━━━━━━━━━ 3043s 938ms/step -